# Camada Bronze — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

Este notebook é responsável por:

1. Criar o catálogo `vcommerce_catalog`, o schema `bronze` e o Volume `vcommerce_vol` (se ainda não existirem).
2. Ler cada arquivo CSV do Volume e gravá-lo como tabela Delta na camada Bronze, acrescentando a coluna `timestamp_ingestion` com o instante exato de carga.
3. Sanitizar nomes de colunas para garantir compatibilidade com o Delta Lake (caracteres especiais substituídos por `_`).
4. Ingestão de API: consumir a cotação do dólar (PTAX) via endpoint público do Banco Central do Brasil e salvar os registros na tabela `bronze.tb_cotacao_dolar`.

---

### Mapeamento de arquivos para tabelas Bronze

| Arquivo Original | Nome da Tabela (Bronze) |
|---|---|
| clientes.csv | bronze.tb_clientes |
| pedidos.csv | bronze.tb_pedidos |
| catalogo_produtos.csv | bronze.tb_catalogo_produtos |
| suporte_tickets.csv | bronze.tb_suporte_tickets |
| clickstream.csv | bronze.tb_clickstream |
| avaliacoes.csv | bronze.tb_avaliacoes |
| *(API BCB/PTAX)* | bronze.tb_cotacao_dolar |

In [0]:
# Parâmetros de data para a coleta da cotação PTAX.
# Podem ser sobrescritos via widgets ou Jobs no Databricks.

from pyspark.sql import functions as F
from datetime import datetime

data_inicio = '01-01-2018'   # V-Commerce fundada em 2018
data_fim    = datetime.today().strftime('%m-%d-%Y')

print(f'Período de cotação PTAX: {data_inicio} - {data_fim}')

In [0]:
# ─── Configuração do catálogo, schema e Volume ───────────────────────────────

catalogo        = 'vcommerce_catalog'
bronze_db_name  = 'vcommerce_bronze'
landing_volume  = 'vcommerce_vol'
volume_path     = f'/Volumes/{catalogo}/{bronze_db_name}/{landing_volume}'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalogo}')
spark.sql(f'USE CATALOG {catalogo}')

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {bronze_db_name}')
spark.sql(f'USE SCHEMA {bronze_db_name}')

spark.sql(f'CREATE VOLUME IF NOT EXISTS {landing_volume}')

print(f'Catálogo  : {catalogo}')
print(f'Schema    : {bronze_db_name}')
print(f'Volume    : {volume_path}')

In [0]:
# ─── Sanitização de nomes de colunas ─────────────────────────────────────────
# O Delta Lake não aceita certos caracteres especiais em nomes de colunas
# (espaços, parênteses, vírgulas, ponto-e-vírgulas, chaves, etc.).
# Esta função substitui qualquer caractere inválido por '_' e remove
# underscores duplicados/iniciais/finais.

import re
from pyspark.sql import DataFrame

def sanitize_column_names(df: DataFrame) -> DataFrame:
    """Renomeia colunas substituindo caracteres inválidos para Delta Lake por '_'."""
    def clean(name: str) -> str:
        # Substitui qualquer char que não seja letra, número ou _ por _
        cleaned = re.sub(r'[^\w]', '_', name, flags=re.UNICODE)
        # Remove underscores duplicados
        cleaned = re.sub(r'_+', '_', cleaned)
        # Remove underscore no início e no fim
        cleaned = cleaned.strip('_')
        return cleaned.lower()

    renamed = {col: clean(col) for col in df.columns if col != clean(col)}
    if renamed:
        print(f'  Colunas renomeadas: {renamed}')
    for old, new in renamed.items():
        df = df.withColumnRenamed(old, new)
    return df

print('Função sanitize_column_names definida.')

In [0]:
# ─── Função principal de ingestão ────────────────────────────────────────────
# Lê um CSV do Volume, sanitiza nomes de colunas, adiciona timestamp_ingestion
# e grava como tabela Delta na camada Bronze.
#
# Opções relevantes:
#   header=true        - primeira linha é o cabeçalho
#   inferSchema=true   - infere tipos automaticamente (Bronze: aceitável)
#   multiLine=true     - tolera quebras de linha dentro de campos de texto
#   escape='"'         - trata aspas escapadas corretamente
#   encoding=UTF-8     - charset padrão dos CSVs
#
# mode=append + overwriteSchema=true:
#   Permite reprocessamento sem perder dados anteriores e tolera
#   evoluções de esquema entre execuções.

def ingest_csv_to_bronze(csv_filename: str, table_name: str) -> None:
    """Lê um CSV do Volume e grava como tabela Delta na camada Bronze."""
    path = f'{volume_path}/{csv_filename}'
    print(f'\nIngerindo: {path}')
    print(f'  bronze.{table_name}')

    df = (
        spark.read
             .option('header',      'true')
             .option('inferSchema', 'true')
             .option('multiLine',   'true')
             .option('escape',      '"')
             .option('encoding',    'UTF-8')
             .csv(path)
    )

    # Sanitiza nomes de colunas antes de gravar no Delta
    df = sanitize_column_names(df)

    # Adiciona marca temporal de ingestão para rastreabilidade
    df = df.withColumn('timestamp_ingestion', F.current_timestamp())

    (
        df.write
          .format('delta')
          .mode('append')
          .option('overwriteSchema', 'true')
          .saveAsTable(f'{bronze_db_name}.{table_name}')
    )

    count = spark.table(f'{bronze_db_name}.{table_name}').count()
    print(f'{df.count()} linhas carregadas (total na tabela: {count})')

print('Função ingest_csv_to_bronze definida.')

In [0]:
# ─── Ingestão dos arquivos CSV do dataset V-Commerce ─────────────────────────
#
# As seis tabelas operacionais do case:
#   1. Clientes          (~60.000 registros)  – perfis cadastrais
#   2. Pedidos           (~310.000 registros) – histórico de compras
#   3. Catálogo Produtos (~500 produtos)      – referência de produtos
#   4. Tickets de Suporte(~35.000 registros)  – interações com SAC
#   5. Clickstream       (~500.000 eventos)   – comportamento digital
#   6. Avaliações        (~150.000 registros) – notas e NPS pós-compra

csv_table_map = [
    ('clientes.csv',           'tb_clientes'),
    ('pedidos.csv',            'tb_pedidos'),
    ('catalogo_produtos.csv',  'tb_catalogo_produtos'),
    ('suporte_tickets.csv',    'tb_suporte_tickets'),
    ('clickstream.csv',        'tb_clickstream'),
    ('avaliacoes.csv',         'tb_avaliacoes'),
]

for csv_file, table in csv_table_map:
    ingest_csv_to_bronze(csv_file, table)

print('\nIngestão de todos os CSVs concluída.')

In [0]:
# ─── Validação rápida pós-ingestão ───────────────────────────────────────────
# Exibe contagem e schema de cada tabela Bronze para confirmar a ingestão.

tables = [t for _, t in csv_table_map]

print('=== Resumo da Camada Bronze ===')
print(f'{"Tabela":<30} {"Linhas":>10} {"Colunas":>8}')
print('-' * 52)

for table in tables:
    df_check = spark.table(f'{bronze_db_name}.{table}')
    n_rows   = df_check.count()
    n_cols   = len(df_check.columns)
    print(f'{bronze_db_name}.{table:<22} {n_rows:>10,} {n_cols:>8}')

In [0]:
# ─── Ingestão da cotação do dólar (BCB / PTAX) ───────────────────────────────
# Consome o endpoint público do Banco Central do Brasil para obter a série
# histórica da cotação PTAX (compra) no período definido nos parâmetros.
# Útil para análises de valor de pedido em dólar e benchmarks internacionais.
#
# Período configurável via parâmetros data_inicio / data_fim (cell-01).

import requests

url = (
    'https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/'
    'CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)'
    f'?@dataInicial=\'{data_inicio}\'&@dataFinalCotacao=\'{data_fim}\''
    '&$select=dataHoraCotacao,cotacaoCompra&$format=json'
)

print(f'Chamando API BCB/PTAX...')
response = requests.get(url, timeout=60)
response.raise_for_status()

registros = response.json().get('value', [])
print(f'Registros recebidos: {len(registros)}')

In [0]:
# ─── Grava cotação PTAX em bronze.tb_cotacao_dolar ───────────────────────────
# Converte a resposta JSON em DataFrame Spark, padroniza tipos e nomes,
# e persiste como tabela Delta.
# Lança exceção se a API retornar lista vazia (período inválido ou fora do ar).

if registros:
    df_cotacao = (
        spark.createDataFrame(registros)
             .withColumnRenamed('dataHoraCotacao', 'data_hora_cotacao')
             .withColumnRenamed('cotacaoCompra',   'cotacao_compra')
             .withColumn('cotacao_compra',    F.col('cotacao_compra').cast('double'))
             .withColumn('data_hora_cotacao', F.to_timestamp('data_hora_cotacao'))
             .withColumn('timestamp_ingestion', F.current_timestamp())
    )

    (
        df_cotacao.write
                  .format('delta')
                  .mode('append')
                  .option('overwriteSchema', 'true')
                  .saveAsTable(f'{bronze_db_name}.tb_cotacao_dolar')
    )

    print(f'bronze.tb_cotacao_dolar gravada com {df_cotacao.count()} registros.')
else:
    raise ValueError(
        'API do Banco Central retornou lista vazia. '
        'Verifique o período informado (data_inicio / data_fim).'
    )

In [0]:
# ─── Listagem final das tabelas Bronze criadas ────────────────────────────────

print('=== Tabelas disponíveis em bronze ===')
spark.sql(f'SHOW TABLES IN {bronze_db_name}').show(truncate=False)